In [1]:
import numpy as np
import pandas as pd
import pygad
import time
from inventory_model import *

def decode(genes):
    m = np.clip(np.array(genes), 0, 1.5).reshape(N, W)
    return m * weekly_fc

def fitness_func(ga_instance, solution, solution_idx):
    return fitness(decode(solution))

baseline = BUDGET_FACTOR * weekly_fc
base_val = simulate_profit(baseline, val_scen).mean()
base_act = simulate_profit(baseline, actual)[0]

def run_ga(seed, generations=300):
    rng = np.random.default_rng(seed)
    pop = rng.uniform(0, 1.5, size=(80, N * W))
    pop[:60] = np.clip(BUDGET_FACTOR + rng.normal(0, 0.05, size=(60, N * W)), 0, 1.5)
    pop[0, :] = BUDGET_FACTOR

    ga = pygad.GA(
        num_generations=generations,
        num_parents_mating=40,
        initial_population=pop,
        fitness_func=fitness_func,
        gene_type=float,
        parent_selection_type="tournament",
        K_tournament=3,
        crossover_type="uniform",
        mutation_type="random",
        mutation_percent_genes=15,
        mutation_by_replacement=False,
        random_mutation_min_val=-0.03,
        random_mutation_max_val=0.03,
        keep_elitism=2,
        random_seed=seed,
    )
    ga.run()
    sol, _, _ = ga.best_solution()
    return decode(sol)

In [2]:
rows = []
for seed in [1, 2, 3, 4, 5]:
    start = time.time()
    plan = run_ga(seed)
    v = simulate_profit(plan, val_scen).mean()
    a = simulate_profit(plan, actual)[0]
    rows.append({
        "seed": seed,
        "train_fitness": fitness(plan),
        "val_mean": v,
        "val_gain_%": (v / base_val - 1) * 100,
        "real_2017": a,
        "real_gain_%": (a / base_act - 1) * 100,
    })
    print(f"Seed {seed} done in {(time.time() - start) / 60:.1f} min | validation gain {rows[-1]['val_gain_%']:.2f}%")

results = pd.DataFrame(rows)
results.to_csv("multi_seed_results.csv", index=False)

Seed 1 done in 2.6 min | validation gain 0.40%
Seed 2 done in 2.7 min | validation gain 0.44%
Seed 3 done in 3.5 min | validation gain 0.42%
Seed 4 done in 3.1 min | validation gain 0.40%
Seed 5 done in 3.6 min | validation gain 0.43%


In [4]:
print(results.round(2).to_string(index=False))
print()
print(f"Validation gain: mean {results['val_gain_%'].mean():.2f}%, "
      f"std {results['val_gain_%'].std():.2f}%, "
      f"min {results['val_gain_%'].min():.2f}%, max {results['val_gain_%'].max():.2f}%")
print(f"Real 2017 gain:  mean {results['real_gain_%'].mean():.2f}%")

 seed  train_fitness  val_mean  val_gain_%  real_2017  real_gain_%
    1      752262.16 752243.69        0.40  750955.75        -0.12
    2      752558.35 752532.05        0.44  751609.91        -0.04
    3      752449.65 752411.30        0.42  751266.12        -0.08
    4      752316.61 752282.30        0.40  752640.78         0.10
    5      752551.56 752510.05        0.43  752402.50         0.07

Validation gain: mean 0.42%, std 0.02%, min 0.40%, max 0.44%
Real 2017 gain:  mean -0.02%
